In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, mean, min, max, round, least, greatest

spark = SparkSession.builder.appName("WeatherAnalysis").getOrCreate()
filepath = "D:/ABD0008/DATA/Weather Data in India from 1901 to 2017.csv"

df = spark.read.csv(filepath, header=True, inferSchema=True)
months = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

In [3]:
avg_expr = sum(col(m) for m in months) / len(months)
df = df.withColumn("Avg_Temp", round(avg_expr, 2))

In [4]:
# Calculate the mean for each month
avg_exprs = [round(mean(col(m)), 2).alias(m) for m in months]
avg_row = df.select(*avg_exprs) \
            .withColumn("YEAR", lit("Average")) \
            .withColumn("Avg_Temp", lit(None))

# Reorder columns to match original DataFrame
avg_row = avg_row.select("YEAR", *months, "Avg_Temp")

# Cast the original YEAR column to string to allow union with the word "Average"
df_str = df.withColumn("YEAR", col("YEAR").cast("string"))
df_with_avg = df_str.unionByName(avg_row)

In [5]:
df_with_avg = df_with_avg.withColumn("Min_Temp", least(*[col(m) for m in months])) \
                         .withColumn("Max_Temp", greatest(*[col(m) for m in months]))

In [6]:
# Remove the 'Average' row and cast YEAR back to integer
df_years = df_with_avg.filter(col("YEAR") != "Average").withColumn("YEAR", col("YEAR").cast("int"))

# Formula to group ranges like 1901-1910 into 1910
df_years = df_years.withColumn("Decade", ((col("YEAR") - 1) / 10).cast("int") * 10 + 10)

decade_df = df_years.groupBy("Decade").agg(
    *[round(mean(col(m)), 2).alias(m) for m in months]
).orderBy("Decade")

In [7]:
hottest_row = df_years.orderBy(col("Avg_Temp").desc()).select("YEAR", "Avg_Temp").first()
coldest_row = df_years.orderBy(col("Avg_Temp").asc()).select("YEAR", "Avg_Temp").first()

print(f"Hottest Year: {hottest_row['YEAR']}")
print(f"Coldest Year: {coldest_row['YEAR']}")

Hottest Year: 2016
Coldest Year: 1917


In [8]:
global_min = df_years.select(min("Min_Temp")).first()[0]
global_max = df_years.select(max("Max_Temp")).first()[0]

year_with_min = df_years.filter(col("Min_Temp") == global_min).select("YEAR").first()[0]
year_with_max = df_years.filter(col("Max_Temp") == global_max).select("YEAR").first()[0]

print(f"Absolute Min Temp ({global_min}°C) recorded in: {year_with_min}")
print(f"Absolute Max Temp ({global_max}°C) recorded in: {year_with_max}")

Absolute Min Temp (17.25°C) recorded in: 1918
Absolute Max Temp (30.78°C) recorded in: 1921


In [9]:
raise_exprs = [round(max(col(m)) - min(col(m)), 2).alias(f"{m}_Raise") for m in months]
raise_df = df_years.select(*raise_exprs)
raise_df.show()

+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|JAN_Raise|FEB_Raise|MAR_Raise|APR_Raise|MAY_Raise|JUN_Raise|JUL_Raise|AUG_Raise|SEP_Raise|OCT_Raise|NOV_Raise|DEC_Raise|
+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+
|     3.67|     5.79|     4.83|     4.72|     3.81|     2.55|     1.99|     1.96|     2.64|     3.72|     3.33|     3.91|
+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+

